In [4]:
import getpass

user = getpass.getuser()

print("시스템 사용자명:", user)

시스템 사용자명: Admin


In [1]:
import sys

print(sys.executable)

c:\Users\Admin\abc\.abc\Scripts\python.exe


In [12]:
from PIL import Image

img = Image.open("cat.jpg")
gray = img.convert("L")

# 1: 흑백 이미지는 0(검정) 또는 255(흰색) 둘 중 한개
#  => 문서에서 글자만 추출하고 싶을 때 (예: OCR)
# L: 0~255 (256단계의 회색)

gray.show()

print(gray.size)
print(gray.mode)

(384, 271)
L


In [14]:
from PIL import Image

img = Image.open("cat.jpg")

img_1 = img.convert("1")
img_L = img.convert("L")

print("크기:", img.size)

print("1 모드:", len(img_1.tobytes()), "bytes")
print("L 모드:", len(img_L.tobytes()), "bytes")

크기: (384, 271)
1 모드: 13008 bytes
L 모드: 104064 bytes


In [ ]:
from PIL import Image
import psutil
import time
import os


def check_image(mode):
    process = psutil.Process(os.getpid())

    # 이미지 열기
    img = Image.open("cat.jpg")

    # 처리 전 RAM
    ram_before = process.memory_info().rss

    # 시간 측정 시작
    start = time.perf_counter()

    # 이미지 처리
    result = img.convert(mode)
    data = result.tobytes()

    # 시간 측정 종료
    end = time.perf_counter()

    # 처리 후 RAM
    ram_after = process.memory_info().rss

    # 결과
    print("Mode:", mode)
    print("Time:", round((end - start) * 1000, 3), "ms")
    print("RAM 전:", round(ram_before / 1024 / 1024, 2), "MB")
    print("RAM 후:", round(ram_after / 1024 / 1024, 2), "MB")
    print("RAM 증가:", round((ram_after - ram_before) / 1024 / 1024, 2), "MB")
    print("이미지 데이터:", round(len(data) / 1024 / 1024, 2), "MB")
    print()

In [24]:
check_image("1")

Mode: 1
Time: 4.069 ms
RAM 전: 79.73 MB
RAM 후: 79.73 MB
RAM 증가: 0.0 MB
이미지 데이터: 0.01 MB



In [25]:
check_image("L")

Mode: L
Time: 0.78 ms
RAM 전: 79.73 MB
RAM 후: 79.74 MB
RAM 증가: 0.0 MB
이미지 데이터: 0.1 MB



In [26]:
check_image("RGB")

Mode: RGB
Time: 1.688 ms
RAM 전: 79.74 MB
RAM 후: 80.43 MB
RAM 증가: 0.69 MB
이미지 데이터: 0.3 MB



In [29]:
from PIL import Image
import psutil
import time
import os
import pandas as pd


def image_simulation(file_name):
    process = psutil.Process(os.getpid())
    original = Image.open(file_name)

    # 테스트할 이미지 크기
    sizes = [
        (320, 240),
        (640, 480),
        (1280, 720),
        (1920, 1080)
    ]

    # 테스트할 이미지 모드
    modes = ["1", "L", "RGB"]

    results = []

    for width, height in sizes:
        for mode in modes:

            ram_before = process.memory_info().rss

            start = time.perf_counter()

            # 크기 변경
            img = original.resize((width, height))

            # 이미지 모드 변경
            img = img.convert(mode)

            # 실제 픽셀 데이터를 메모리에 생성
            data = img.tobytes()

            end = time.perf_counter()

            ram_after = process.memory_info().rss

            time_ms = (end - start) * 1000
            ram_mb = (ram_after - ram_before) / 1024 / 1024
            data_mb = len(data) / 1024 / 1024

            results.append({
                "Width": width,
                "Height": height,
                "Mode": mode,
                "Time(ms)": round(time_ms, 3),
                "RAM 증가(MB)": round(ram_mb, 3),
                "픽셀 데이터(MB)": round(data_mb, 3)
            })

    return pd.DataFrame(results)


df = image_simulation("cat.jpg")

print(df)

    Width  Height Mode  Time(ms)  RAM 증가(MB)  픽셀 데이터(MB)
0     320     240    1     4.127       1.113       0.009
1     320     240    L     1.742       0.293       0.073
2     320     240  RGB     1.970       0.000       0.220
3     640     480    1     8.614       0.512       0.037
4     640     480    L     5.497       0.121       0.293
5     640     480  RGB     4.934       1.629       0.879
6    1280     720    1    22.969      -1.168       0.110
7    1280     720    L     8.661       1.312       0.879
8    1280     720  RGB    13.418       3.707       2.637
9    1920    1080    1    71.377      -4.180       0.247
10   1920    1080    L    33.727       2.160       1.978
11   1920    1080  RGB    37.606       9.664       5.933
